# Data story — Amazon Baby Products

Notebook này trả lời trực tiếp phản hồi của giảng viên ngày 03/09: dữ liệu có sạch không, phân bố ra sao, long-tail/mất cân bằng mạnh đến đâu, và temporal split giữ lại population nào.

Notebook quét **toàn bộ** raw file và frozen training graph; không lấy mẫu để tạo số liệu chính. Nó không train model và không tạo claim về NDCG hay sampler.

Output cuối gồm bốn hình, một JSON có assertion, một bản tóm tắt Markdown và một file ZIP để gửi lại cho Codex.

In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

# Nếu thư mục Drive của bạn khác, chỉ sửa dòng này.
DRIVE_ROOT = Path('/content/drive/MyDrive')
PROJECT_DATA = DRIVE_ROOT / 'Phase2_Amazon_Audit'
RAW_PATH = PROJECT_DATA / 'raw' / 'Baby_Products.csv.gz'
AUDIT_PATH = PROJECT_DATA / 'audit_outputs' / 'Baby_Products_protocol_audit.json'
ARTIFACT_DIR = PROJECT_DATA / 'g2c_baby_p4'
MANIFEST_PATH = ARTIFACT_DIR / 'baby_p4_g2c_manifest.json'
TRAIN_PATH = ARTIFACT_DIR / 'baby_p4_train_edges.csv.gz'
OUTPUT_DIR = PROJECT_DATA / 'data_story'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_SOURCE_SHA256 = 'e2a8d0498afed767ee2615db7fac549559d82490b1a73c7241b84b5e9e8c279e'

for path in (RAW_PATH, AUDIT_PATH, MANIFEST_PATH, TRAIN_PATH):
    assert path.is_file(), f'Thiếu file: {path}'

print('Raw:', RAW_PATH)
print('Training graph:', TRAIN_PATH)
print('Output:', OUTPUT_DIR)

## Câu hỏi và ranh giới

1. Có missing, parse error, duplicate hay rating ngoài miền không?
2. Rating và số interaction theo thời gian phân bố thế nào?
3. User/item degree có long-tail không; top item giữ bao nhiêu interaction?
4. Temporal split loại bao nhiêu target vì cold-start?

`Semantic diversity` không được tính: artifact pure-ID không có title, category hoặc text. Đây là giới hạn dữ liệu, không phải giá trị bằng 0.

In [ ]:
from array import array
from collections import Counter
from pathlib import Path
import csv
import datetime as dt
import gzip
import hashlib
import json
import math


def sha256_file(path, block_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(block_size), b''):
            digest.update(block)
    return digest.hexdigest()


def open_csv(path):
    path = Path(path)
    if path.name.endswith('.gz'):
        return gzip.open(path, 'rt', encoding='utf-8', newline='')
    return path.open('rt', encoding='utf-8', newline='')


def month_key(timestamp_ms):
    value = dt.datetime.fromtimestamp(timestamp_ms / 1000, tz=dt.timezone.utc)
    return f'{value.year:04d}-{value.month:02d}'


def scan_raw(path, progress_every=1_000_000):
    raw_rows = 0
    parsed_rows = 0
    missing_or_invalid = 0
    out_of_range = 0
    p4_rows = 0
    rating_counts = Counter()
    monthly_clean = Counter()
    monthly_p4 = Counter()

    with open_csv(path) as handle:
        reader = csv.DictReader(handle)
        expected = ['user_id', 'parent_asin', 'rating', 'timestamp']
        if list(reader.fieldnames or []) != expected:
            raise ValueError(f'Schema không khớp: {reader.fieldnames}')
        for row in reader:
            raw_rows += 1
            if progress_every and raw_rows % progress_every == 0:
                print(f'  raw: {raw_rows:,} rows')
            user_id = (row.get('user_id') or '').strip()
            item_id = (row.get('parent_asin') or '').strip()
            try:
                rating = float(row.get('rating', ''))
                timestamp_ms = int(row.get('timestamp', ''))
            except (TypeError, ValueError):
                missing_or_invalid += 1
                continue
            if not user_id or not item_id:
                missing_or_invalid += 1
                continue
            parsed_rows += 1
            rating_counts[f'{rating:g}'] += 1
            if not 1.0 <= rating <= 5.0:
                out_of_range += 1
                continue
            month = month_key(timestamp_ms)
            monthly_clean[month] += 1
            if rating >= 4.0:
                p4_rows += 1
                monthly_p4[month] += 1

    return {
        'raw_rows': raw_rows,
        'parsed_rows': parsed_rows,
        'missing_or_invalid': missing_or_invalid,
        'out_of_range': out_of_range,
        'p4_rows': p4_rows,
        'rating_counts': dict(sorted(rating_counts.items(), key=lambda pair: float(pair[0]))),
        'monthly_clean': dict(sorted(monthly_clean.items())),
        'monthly_p4': dict(sorted(monthly_p4.items())),
    }


def degree_histogram(values):
    return dict(sorted(Counter(values).items()))


def quantile_from_histogram(histogram, probability):
    count = sum(histogram.values())
    if count == 0:
        return 0.0
    target = probability * (count - 1)
    lower_rank = math.floor(target)
    upper_rank = math.ceil(target)

    def value_at(rank):
        cumulative = 0
        for value, frequency in sorted(histogram.items()):
            cumulative += frequency
            if rank < cumulative:
                return float(value)
        raise AssertionError('Không tìm được quantile')

    lower = value_at(lower_rank)
    upper = value_at(upper_rank)
    return lower + (target - lower_rank) * (upper - lower)


def gini_from_histogram(histogram):
    n = sum(histogram.values())
    total = sum(value * frequency for value, frequency in histogram.items())
    if n == 0 or total == 0:
        return 0.0
    weighted_rank_sum = 0.0
    rank_before = 0
    for value, frequency in sorted(histogram.items()):
        first_rank = rank_before + 1
        last_rank = rank_before + frequency
        rank_sum = frequency * (first_rank + last_rank) / 2
        weighted_rank_sum += value * rank_sum
        rank_before = last_rank
    return (2 * weighted_rank_sum) / (n * total) - (n + 1) / n


def ccdf_from_histogram(histogram):
    total = sum(histogram.values())
    remaining = total
    x_values, y_values = [], []
    for degree, frequency in sorted(histogram.items()):
        x_values.append(degree)
        y_values.append(remaining / total if total else 0.0)
        remaining -= frequency
    return x_values, y_values


def top_interaction_shares(degrees, fractions=(0.01, 0.05, 0.10, 0.20)):
    ordered = sorted(degrees, reverse=True)
    total = sum(ordered)
    shares = {}
    for fraction in fractions:
        count = max(1, math.ceil(len(ordered) * fraction))
        shares[f'top_{int(fraction * 100)}pct_items'] = sum(ordered[:count]) / total if total else 0.0
    return shares


def scan_training_graph(path, num_users, num_items, progress_every=1_000_000):
    user_degrees = array('I', [0]) * num_users
    item_degrees = array('I', [0]) * num_items
    edge_count = 0
    with open_csv(path) as handle:
        reader = csv.DictReader(handle)
        required = {'user_idx', 'item_idx', 'rating', 'timestamp_ms', 'source_row'}
        if set(reader.fieldnames or []) != required:
            raise ValueError(f'Schema train không khớp: {reader.fieldnames}')
        for row in reader:
            user_idx = int(row['user_idx'])
            item_idx = int(row['item_idx'])
            if not 0 <= user_idx < num_users or not 0 <= item_idx < num_items:
                raise AssertionError('Mapped ID nằm ngoài miền đã freeze')
            user_degrees[user_idx] += 1
            item_degrees[item_idx] += 1
            edge_count += 1
            if progress_every and edge_count % progress_every == 0:
                print(f'  train: {edge_count:,} edges')

    user_hist = degree_histogram(user_degrees)
    item_hist = degree_histogram(item_degrees)
    return {
        'edge_count': edge_count,
        'nonzero_users': sum(degree > 0 for degree in user_degrees),
        'nonzero_items': sum(degree > 0 for degree in item_degrees),
        'user_histogram': user_hist,
        'item_histogram': item_hist,
        'user_singleton_rate': user_hist.get(1, 0) / num_users,
        'item_singleton_rate': item_hist.get(1, 0) / num_items,
        'user_p50': quantile_from_histogram(user_hist, 0.50),
        'user_p90': quantile_from_histogram(user_hist, 0.90),
        'user_p99': quantile_from_histogram(user_hist, 0.99),
        'item_p50': quantile_from_histogram(item_hist, 0.50),
        'item_p90': quantile_from_histogram(item_hist, 0.90),
        'item_p99': quantile_from_histogram(item_hist, 0.99),
        'item_degree_gini': gini_from_histogram(item_hist),
        'top_item_interaction_shares': top_interaction_shares(item_degrees),
    }


def build_summary(source_sha256, audit, manifest, raw, train):
    expected_input = manifest['input_counts']
    expected_graph = manifest['training_graph']
    assertions = {
        'source_sha256_matches_registered': source_sha256 == EXPECTED_SOURCE_SHA256,
        'audit_and_manifest_source_match': audit['sha256'] == manifest['source']['sha256'] == source_sha256,
        'raw_rows_match_manifest': raw['raw_rows'] == expected_input['raw_rows'],
        'p4_rows_match_manifest': raw['p4_rows'] == expected_input['p4_rows_before_pair_dedup'],
        'train_edges_match_manifest': train['edge_count'] == expected_graph['edges'],
        'train_users_match_manifest': train['nonzero_users'] == expected_graph['users'],
        'train_items_match_manifest': train['nonzero_items'] == expected_graph['items'],
        'degree_sums_match_edges': (
            sum(degree * frequency for degree, frequency in train['user_histogram'].items())
            == sum(degree * frequency for degree, frequency in train['item_histogram'].items())
            == train['edge_count']
        ),
    }
    if not all(assertions.values()):
        failed = [name for name, passed in assertions.items() if not passed]
        raise AssertionError(f'Bằng chứng không khớp: {failed}')

    return {
        'status': 'FULL_DATA_EDA_EXECUTED_AND_RECONCILED',
        'scope': 'Amazon Reviews 2023 Baby_Products, pure-ID, P4 temporal warm-start',
        'source': {
            'sha256': source_sha256,
            'raw_rows': raw['raw_rows'],
            'parsed_rows': raw['parsed_rows'],
            'missing_or_invalid': raw['missing_or_invalid'],
            'out_of_range_rating_rows': raw['out_of_range'],
            'duplicate_user_item_rows': audit['duplicate_user_item_rows'],
            'rating_counts': raw['rating_counts'],
            'monthly_clean': raw['monthly_clean'],
            'monthly_p4': raw['monthly_p4'],
        },
        'positive_policy': {
            'definition': 'P4: rating >= 4 and rating <= 5',
            'p4_rows': raw['p4_rows'],
            'retention_from_parsed_rows': raw['p4_rows'] / raw['parsed_rows'],
        },
        'training_graph': {
            'edges': train['edge_count'],
            'users': train['nonzero_users'],
            'items': train['nonzero_items'],
            'density': expected_graph['density'],
            'components': expected_graph['components'],
            'user_degree': {
                'p50': train['user_p50'], 'p90': train['user_p90'], 'p99': train['user_p99'],
                'singleton_rate': train['user_singleton_rate'],
                'histogram': {str(k): v for k, v in train['user_histogram'].items()},
            },
            'item_degree': {
                'p50': train['item_p50'], 'p90': train['item_p90'], 'p99': train['item_p99'],
                'singleton_rate': train['item_singleton_rate'],
                'gini': train['item_degree_gini'],
                'top_item_interaction_shares': train['top_item_interaction_shares'],
                'histogram': {str(k): v for k, v in train['item_histogram'].items()},
            },
        },
        'temporal_population': manifest['partitions'],
        'semantic_diversity': {
            'status': 'NOT_MEASURABLE_FROM_CURRENT_ARTIFACT',
            'reason': 'Pure-ID rating-only data has no title, category, text, taxonomy, or semantic embedding.',
        },
        'assertions': assertions,
        'claim_boundary': 'EDA only; no recommender quality, sampler superiority, cold-start, or semantic-diversity claim.',
    }

## Chạy full scan và đối soát

Cell này đọc raw file một lần, đọc training graph một lần và kiểm tra lại các count trọng yếu. Không tiếp tục nếu checksum hoặc count lệch với manifest.

In [ ]:
audit = json.loads(AUDIT_PATH.read_text(encoding='utf-8'))
manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))

print('1/3 Kiểm tra SHA-256...')
source_sha256 = sha256_file(RAW_PATH)
print('2/3 Quét toàn bộ raw data...')
raw = scan_raw(RAW_PATH)
print('3/3 Quét frozen training graph...')
train = scan_training_graph(
    TRAIN_PATH,
    num_users=int(manifest['training_graph']['users']),
    num_items=int(manifest['training_graph']['items']),
)
summary = build_summary(source_sha256, audit, manifest, raw, train)
print(json.dumps({
    'status': summary['status'],
    'assertions': summary['assertions'],
    'raw_rows': summary['source']['raw_rows'],
    'p4_rows': summary['positive_policy']['p4_rows'],
    'training_graph': {k: summary['training_graph'][k] for k in ('edges', 'users', 'items')},
}, ensure_ascii=False, indent=2))

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

plt.style.use('seaborn-v0_8-whitegrid')
COLORS = {'all': '#64748b', 'p4': '#2563eb', 'user': '#7c3aed', 'item': '#ea580c', 'warm': '#059669', 'excluded': '#cbd5e1'}

def save_figure(fig, filename):
    path = OUTPUT_DIR / filename
    fig.savefig(path, dpi=180, bbox_inches='tight', facecolor='white')
    plt.show()
    print('Saved:', path)
    return path

# Hình 1: rating và timeline
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
rating_order = sorted(summary['source']['rating_counts'], key=float)
rating_values = [summary['source']['rating_counts'][key] for key in rating_order]
rating_colors = ['#dc2626' if not 1 <= float(key) <= 5 else COLORS['p4'] if float(key) >= 4 else COLORS['all'] for key in rating_order]
bars = axes[0].bar(rating_order, rating_values, color=rating_colors)
axes[0].set_title('Phân bố rating — toàn bộ raw data')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Số interaction')
axes[0].ticklabel_format(axis='y', style='plain')
for bar, value in zip(bars, rating_values):
    axes[0].text(bar.get_x() + bar.get_width()/2, value, f'{value/1e6:.2f}M', ha='center', va='bottom', fontsize=8)
axes[0].text(0.98, 0.82, f"Ngoài miền [1,5]: {summary['source']['out_of_range_rating_rows']:,} dòng", transform=axes[0].transAxes, ha='right', color='#dc2626', fontsize=9)

months = sorted(summary['source']['monthly_clean'])
dates = [dt.datetime.strptime(month, '%Y-%m') for month in months]
axes[1].plot(dates, [summary['source']['monthly_clean'][m] for m in months], label='Rating 1–5', color=COLORS['all'], linewidth=1.6)
axes[1].plot(dates, [summary['source']['monthly_p4'].get(m, 0) for m in months], label='P4 (rating ≥ 4)', color=COLORS['p4'], linewidth=1.6)
for cutoff, label in [(manifest['policy']['t1_ms'], 't1'), (manifest['policy']['t2_ms'], 't2')]:
    cutoff_date = dt.datetime.fromtimestamp(cutoff/1000, tz=dt.timezone.utc).replace(tzinfo=None)
    axes[1].axvline(cutoff_date, color='#dc2626', linestyle='--', alpha=0.7)
    axes[1].text(cutoff_date, axes[1].get_ylim()[1]*0.93, label, color='#dc2626', ha='center')
axes[1].set_title('Interaction theo tháng và temporal cutoff')
axes[1].set_xlabel('Thời gian')
axes[1].set_ylabel('Số interaction/tháng')
axes[1].legend()
axes[1].text(0.98, 0.04, 'Dữ liệu kết thúc 09/2023; không diễn giải phần giảm cuối kỳ như nhu cầu.', transform=axes[1].transAxes, ha='right', fontsize=8, color='#59636e')
fig.suptitle('Hình 1 — P4 là phép biến đổi từ rating sang positive edge', fontsize=14, fontweight='bold')
fig.tight_layout()
figure_1 = save_figure(fig, '01_rating_timeline.png')

# Hình 2: degree CCDF
fig, ax = plt.subplots(figsize=(8.5, 5.5))
user_hist = {int(k): v for k, v in summary['training_graph']['user_degree']['histogram'].items()}
item_hist = {int(k): v for k, v in summary['training_graph']['item_degree']['histogram'].items()}
ux, uy = ccdf_from_histogram(user_hist)
ix, iy = ccdf_from_histogram(item_hist)
ax.loglog(ux, uy, marker='.', markersize=4, linewidth=1.3, label=f"User — singleton {summary['training_graph']['user_degree']['singleton_rate']:.1%}", color=COLORS['user'])
ax.loglog(ix, iy, marker='.', markersize=4, linewidth=1.3, label=f"Item — singleton {summary['training_graph']['item_degree']['singleton_rate']:.1%}", color=COLORS['item'])
ax.set_title('Hình 2 — Long-tail của frozen training graph')
ax.set_xlabel('Degree k (log scale)')
ax.set_ylabel('P(degree ≥ k) (log scale)')
ax.legend()
ax.text(0.02, 0.03, 'Hệ quả: aggregate metric có thể che khuất head/tail behavior.', transform=ax.transAxes, fontsize=9)
fig.tight_layout()
figure_2 = save_figure(fig, '02_degree_long_tail.png')

# Hình 3: popularity concentration
shares = summary['training_graph']['item_degree']['top_item_interaction_shares']
labels = ['Top 1%', 'Top 5%', 'Top 10%', 'Top 20%']
values = [shares['top_1pct_items'], shares['top_5pct_items'], shares['top_10pct_items'], shares['top_20pct_items']]
fig, ax = plt.subplots(figsize=(8.5, 5.2))
bars = ax.bar(labels, values, color=COLORS['item'])
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_ylim(0, min(1.0, max(values) * 1.18))
ax.set_ylabel('Tỷ lệ training interaction')
ax.set_title(f"Hình 3 — Mức tập trung popularity (Gini = {summary['training_graph']['item_degree']['gini']:.3f})")
for bar, value in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, value, f'{value:.1%}', ha='center', va='bottom', fontweight='bold')
fig.text(0.5, 0.02, 'Hệ quả: phải báo exposure theo popularity, không chỉ NDCG tổng.', ha='center', fontsize=9)
fig.tight_layout(rect=[0, 0.06, 1, 1])
figure_3 = save_figure(fig, '03_item_concentration.png')

# Hình 4: temporal population
partitions = summary['temporal_population']
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8))
candidate_counts = [summary['training_graph']['edges'], partitions['validation']['candidate_rows'], partitions['test']['candidate_rows']]
bars = axes[0].bar(['Train edges', 'Validation\ncandidate', 'Test\ncandidate'], candidate_counts, color=[COLORS['p4'], COLORS['all'], COLORS['all']])
axes[0].set_title('Số interaction theo temporal partition')
axes[0].set_ylabel('Số dòng')
for bar, value in zip(bars, candidate_counts):
    axes[0].text(bar.get_x()+bar.get_width()/2, value, f'{value:,}', ha='center', va='bottom', fontsize=8)

names = ['Validation', 'Test']
warm_rates = [partitions['validation']['warm_retention'], partitions['test']['warm_retention']]
excluded_rates = [1-rate for rate in warm_rates]
axes[1].bar(names, warm_rates, label='Warm target giữ lại', color=COLORS['warm'])
axes[1].bar(names, excluded_rates, bottom=warm_rates, label='Loại do OOV', color=COLORS['excluded'])
axes[1].yaxis.set_major_formatter(PercentFormatter(1.0))
axes[1].set_ylim(0, 1)
axes[1].set_title('Population đánh giá sau training-only mapping')
for index, rate in enumerate(warm_rates):
    axes[1].text(index, rate/2, f'{rate:.1%}\nwarm', ha='center', va='center', color='white', fontweight='bold')
    axes[1].text(index, rate + (1-rate)/2, f'{1-rate:.1%}\nOOV', ha='center', va='center', color='#334155', fontweight='bold')
fig.suptitle('Hình 4 — Kết luận chỉ áp dụng cho warm-start population', fontsize=14, fontweight='bold')
fig.tight_layout()
figure_4 = save_figure(fig, '04_temporal_population.png')

In [ ]:
import zipfile

summary_path = OUTPUT_DIR / 'data_story_summary.json'
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2, sort_keys=True) + '\n', encoding='utf-8')

def pct(value):
    return f'{value * 100:.2f}%'

shares = summary['training_graph']['item_degree']['top_item_interaction_shares']
validation = summary['temporal_population']['validation']
test = summary['temporal_population']['test']
story = f'''# Data story — Amazon Baby Products

## Kết luận chính

Dữ liệu đủ lớn và sạch để nghiên cứu graph sampling, nhưng rất thưa và mất cân bằng. Bài toán hiện tại chỉ đánh giá warm-start recommendation; không đại diện cho cold-start.

## 1. Chất lượng và interaction semantics

- Raw rows: {summary['source']['raw_rows']:,}.
- Missing/parse-invalid: {summary['source']['missing_or_invalid']:,}.
- Rating ngoài [1,5]: {summary['source']['out_of_range_rating_rows']:,}.
- Duplicate user–item: {summary['source']['duplicate_user_item_rows']:,}.
- P4 rows (rating >= 4): {summary['positive_policy']['p4_rows']:,}, giữ {pct(summary['positive_policy']['retention_from_parsed_rows'])} parsed rows.

**Hệ quả:** P4 là project transformation từ explicit rating sang implicit positive, không phải thuộc tính có sẵn của nguồn.

## 2. Frozen training graph

- {summary['training_graph']['edges']:,} edge; {summary['training_graph']['users']:,} user; {summary['training_graph']['items']:,} item.
- User degree p50/p90/p99: {summary['training_graph']['user_degree']['p50']:.0f}/{summary['training_graph']['user_degree']['p90']:.0f}/{summary['training_graph']['user_degree']['p99']:.0f}; singleton {pct(summary['training_graph']['user_degree']['singleton_rate'])}.
- Item degree p50/p90/p99: {summary['training_graph']['item_degree']['p50']:.0f}/{summary['training_graph']['item_degree']['p90']:.0f}/{summary['training_graph']['item_degree']['p99']:.0f}; singleton {pct(summary['training_graph']['item_degree']['singleton_rate'])}.
- Largest connected component: {pct(summary['training_graph']['components']['largest_component_fraction'])} số node.

**Hệ quả:** phần lớn graph có đường liên kết cho message passing, nhưng degree imbalance rất mạnh; model/sampler phải được phân tích theo popularity cohort, không chỉ mean degree hoặc aggregate NDCG.

## 3. Popularity concentration

- Item-degree Gini: {summary['training_graph']['item_degree']['gini']:.4f}.
- Top 1% item giữ {pct(shares['top_1pct_items'])} training interaction.
- Top 5% item giữ {pct(shares['top_5pct_items'])}.
- Top 10% item giữ {pct(shares['top_10pct_items'])}.
- Top 20% item giữ {pct(shares['top_20pct_items'])}.

**Hệ quả:** khi có recommendation output, phải báo catalog coverage và head/middle/tail exposure để phát hiện gain do popularity concentration.

## 4. Temporal population

- Validation: giữ {validation['warm_rows']:,}/{validation['candidate_rows']:,} warm target ({pct(validation['warm_retention'])}).
- Test: giữ {test['warm_rows']:,}/{test['candidate_rows']:,} warm target ({pct(test['warm_retention'])}).

**Hệ quả:** headline result chỉ mô tả user và item đã xuất hiện trong training graph. Cold-start nằm ngoài claim.

## 5. Giới hạn

Pure-ID rating-only không hỗ trợ semantic similarity hoặc intra-list semantic diversity. Muốn đo hai đại lượng này phải bổ sung metadata/category/text bằng một quyết định dữ liệu riêng.

## Integrity

Tất cả assertion checksum/count/degree reconciliation đều pass: {all(summary['assertions'].values())}.
'''
story_path = OUTPUT_DIR / 'DATA_STORY_vn.md'
story_path.write_text(story, encoding='utf-8')

bundle_path = OUTPUT_DIR / 'data_story_bundle.zip'
with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in [figure_1, figure_2, figure_3, figure_4, summary_path, story_path]:
        archive.write(path, arcname=path.name)

print(story)
print('Bundle:', bundle_path)

## Việc của bạn sau khi chạy

1. Kiểm tra cell full scan kết thúc với mọi assertion là `true`.
2. Chạy cell cuối để tạo `data_story_bundle.zip`.
3. Tải ZIP về và gửi nguyên file cho Codex; không cần chụp từng biểu đồ.
4. Nếu cell lỗi, gửi nguyên traceback cùng tên cell; không tự sửa count hoặc bỏ assertion.

In [ ]:
# Chạy cell này trên Colab để tải bundle về máy.
from google.colab import files
files.download(str(bundle_path))